# LeetCode 2878 — Get the Size of a DataFrame bằng Python (pandas)

Link đề: https://leetcode.com/problems/get-the-size-of-a-dataframe/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def getDataframeSize(players: pd.DataFrame) -> List[int]:
    ...
```

Ghi chú: bảng ví dụ (các cột và giá trị cụ thể) trong notebook được viết theo trí nhớ về đề; kết quả `[5, 5]` nên **đối chiếu lại với đề gốc** nếu bạn muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho DataFrame `players`, trả về một **list hai số nguyên** `[số hàng, số cột]`.
2. Hiểu rõ **`shape` = `(hàng, cột)`** và phân biệt với `size` (tổng số ô), `len(df)` (số hàng), `len(df.columns)` (số cột), `count()` (đếm ô **không thiếu**).
3. Đi từ cách **đếm thủ công bằng vòng lặp** → **`len` hai lần** → **`players.shape`** → **`index.size` / `columns.size`** → **`axes` / `to_numpy().shape`**.
4. Biết **trace `shape` qua từng bước biến đổi** (`head`, chọn cột, `drop`, chuyển vị `.T`) để hiểu hình dạng bảng thay đổi ra sao.
5. Tránh các lỗi kinh điển: trả về **tuple** thay vì list, **đảo hàng ↔ cột**, dùng `df.size`, dùng `count()` khi có `NaN`.

## 2. Đề bài & ví dụ

Cho một DataFrame `players` với các cột như `player_id`, `name`, `age`, `position`, `team`.

Hãy viết hàm trả về **kích thước** của DataFrame dưới dạng một **list** `[number_of_rows, number_of_columns]`.

**Ví dụ**

- Input: DataFrame `players`

| player_id | name | age | position | team |
|---|---|---|---|---|
| 846 | Mason | 21 | Forward | Real Madrid |
| 749 | Riley | 30 | Winger | Barcelona |
| 155 | Bob | 28 | Striker | Manchester United |
| 426 | Isabella | 32 | Goalkeeper | Liverpool |
| 875 | Zane | 36 | Midfielder | Bayern Munich |

- Output: `[5, 5]`
- Giải thích: bảng có **5 hàng** và **5 cột**.

(Các giá trị cụ thể ở bảng trên: đối chiếu lại với đề gốc. Điều quan trọng là thứ tự `[hàng, cột]`.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

players = pd.DataFrame({
    "player_id": [846, 749, 155, 426, 875],
    "name": ["Mason", "Riley", "Bob", "Isabella", "Zane"],
    "age": [21, 30, 28, 32, 36],
    "position": ["Forward", "Winger", "Striker", "Goalkeeper", "Midfielder"],
    "team": ["Real Madrid", "Barcelona", "Manchester United", "Liverpool", "Bayern Munich"],
})
print(players)

   player_id      name  age    position               team
0        846     Mason   21     Forward        Real Madrid
1        749     Riley   30      Winger          Barcelona
2        155       Bob   28     Striker  Manchester United
3        426  Isabella   32  Goalkeeper          Liverpool
4        875      Zane   36  Midfielder      Bayern Munich


## 3. Kiến thức nền

### 3.1. Hàng, cột và chiều của bảng

Một DataFrame có **hai chiều (axis)**:

- `axis 0` = chiều **hàng** (đi xuống), chứa các bản ghi (mỗi cầu thủ một hàng);
- `axis 1` = chiều **cột** (đi ngang), chứa các thuộc tính (tên, tuổi, ...).

"Kích thước" của bảng luôn nói theo thứ tự **(hàng, cột)** — giống ma trận `m × n`.

### 3.2. `df.shape`

`df.shape` là một **tuple** `(số hàng, số cột)`. Tuple khác list: tuple dùng `()` và **không sửa được**; `(5, 5) == [5, 5]` là `False`.

In [2]:
print(players.shape)               # (5, 5)
print(type(players.shape))         # <class 'tuple'>
print(players.shape[0], players.shape[1])    # 5 5   (hàng, cột)
print((5, 5) == [5, 5])            # False! tuple khác list
print(list(players.shape))         # [5, 5]

(5, 5)
<class 'tuple'>
5 5
False
[5, 5]


### 3.3. `len(df)` và `len(df.columns)`

- `len(df)` = **số hàng** (độ dài của index).
- `df.columns` là danh sách tên cột; `len(df.columns)` = **số cột**.

In [3]:
print(len(players))                  # 5   (số hàng)
print(players.columns.tolist())      # ['player_id', 'name', 'age', 'position', 'team']
print(len(players.columns))          # 5   (số cột)

5
['player_id', 'name', 'age', 'position', 'team']
5


### 3.4. `df.size`: tổng số ô (không phải số hàng!)

`df.size = số hàng × số cột`. Bảng `5 × 5` có `size = 25`. Đây là một chỗ **rất dễ nhầm**.

In [4]:
print(players.size)                  # 25
print(players.shape[0] * players.shape[1])    # 25

25
25


### 3.5. `df.index`, `df.columns`, `df.axes`

- `df.index.size` / `df.columns.size` cho độ dài của từng trục.
- `df.axes` = `[index, columns]` (list hai trục) — độ dài hai phần tử chính là `[hàng, cột]`.

In [5]:
print(players.index.size, players.columns.size)       # 5 5
print([len(ax) for ax in players.axes])               # [5, 5]

5 5
[5, 5]


### 3.6. `df.count()` đếm ô **không thiếu**

`df.count()` trả về một **Series**: với mỗi cột, số ô **không phải `NaN`**. Nên **không** dùng nó để đếm hàng khi bảng có giá trị thiếu.

In [6]:
df_nan = pd.DataFrame({"a": [1, 2, np.nan], "b": [4, np.nan, np.nan]})
print(df_nan)
print(df_nan.count())                # a: 2, b: 1  (không phải 3, 3)
print(df_nan.shape)                  # (3, 2)

     a    b
0  1.0  4.0
1  2.0  NaN
2  NaN  NaN
a    2
b    1
dtype: int64
(3, 2)


### 3.7. Bảng rỗng vẫn có `shape`

Bảng có cột nhưng chưa có hàng: `shape = (0, số_cột)`. Bảng hoàn toàn rỗng: `(0, 0)`.

In [7]:
empty_rows = pd.DataFrame(columns=["a", "b", "c"])
print(empty_rows.shape)              # (0, 3)
print(pd.DataFrame().shape)          # (0, 0)

(0, 3)
(0, 0)


### 3.8. Vòng lặp trên DataFrame

- `for row in df.itertuples(index=False)` duyệt từng **hàng** (mỗi hàng là một namedtuple).
- `for name in df.columns` duyệt từng **tên cột**.
- `sum(1 for _ in ...)` là cách đếm số phần tử của một bộ lặp mà không lưu lại.

In [8]:
rows_count = sum(1 for _ in players.itertuples(index=False))
cols_count = sum(1 for _ in players.columns)
print(rows_count, cols_count)        # 5 5

5 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Nhìn bảng ở mục 2:

- Đếm **hàng dữ liệu** (không tính dòng tiêu đề): Mason, Riley, Bob, Isabella, Zane → **5 hàng**.
- Đếm **tên cột** ở dòng tiêu đề: `player_id`, `name`, `age`, `position`, `team` → **5 cột**.

Đáp án theo thứ tự `[hàng, cột] = [5, 5]`.

Hãy thử một bảng không vuông để thấy thứ tự quan trọng: bảng gồm **3 hàng, 2 cột** → `[3, 2]` (không phải `[2, 3]`). Mẹo nhớ: "**hàng trước, cột sau**", đọc như chữ viết từ trên xuống rồi từ trái sang phải.

Lưu ý: dòng tiêu đề **không** phải là một hàng dữ liệu; còn cột index (nhãn `0..4`) **cũng không phải** là một cột.

In [9]:
t = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(t)
print(list(t.shape))     # [3, 2]  (3 hàng, 2 cột)

   x  y
0  1  4
1  2  5
2  3  6
[3, 2]


## 5. Cách 1 — Brute Force / baseline: đếm thủ công bằng vòng lặp

Cách "ngây thơ" nhất: **tự đếm** từng hàng và từng cột bằng vòng `for`.

In [10]:
def size_by_counting(players: pd.DataFrame) -> List[int]:
    rows = 0
    for _ in players.itertuples(index=False):      # mỗi lần lặp = một hàng
        rows += 1
    cols = 0
    for _ in players.columns:                      # mỗi lần lặp = một tên cột
        cols += 1
    return [rows, cols]

print(size_by_counting(players))                   # [5, 5]
print(size_by_counting(t))                         # [3, 2]
print(size_by_counting(pd.DataFrame(columns=["a", "b", "c"])))   # [0, 3]

[5, 5]
[3, 2]
[0, 3]


**Độ phức tạp:** duyệt `m` hàng và `c` cột nên **Time `O(m + c)`** (cách này còn dựng namedtuple cho từng hàng nên khá nặng), **Space `O(1)`**.

**Nút thắt:** pandas **đã lưu sẵn** kích thước nên ta không cần đếm lại. Mọi cách sau chỉ **đọc** thông tin có sẵn.

## 6. Ý tưởng thuật toán / bước đột phá

DataFrame lưu số hàng (độ dài `index`) và số cột (độ dài `columns`) như **thuộc tính**. Việc đọc chúng là `O(1)`, không phụ thuộc kích thước bảng:

- `len(df)` → số hàng,
- `len(df.columns)` → số cột,
- `df.shape` → cả hai cùng lúc dưới dạng tuple.

Chỉ còn một "chi tiết đề": kiểu trả về là **list**, nên đổi tuple thành list: `list(df.shape)`.

## 7. Các cách tiếp theo

### Cách 2 — `len` hai lần

**Giải thích.** `len(players)` cho số hàng; `len(players.columns)` cho số cột. Gom vào một list.

In [11]:
def size_len(players: pd.DataFrame) -> List[int]:
    return [len(players), len(players.columns)]

print(size_len(players))      # [5, 5]

[5, 5]


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def getDataframeSize(players: pd.DataFrame) -> List[int]:
    return [len(players), len(players.columns)]

print(getDataframeSize(players))

[5, 5]


**Độ phức tạp Cách 2:** **Time `O(1)`**, **Space `O(1)`**.

---

### Cách 3 — `list(players.shape)` (cách chuẩn)

**Giải thích.** `players.shape` là `(5, 5)`; `list(...)` đổi thành `[5, 5]`. Các phần tử là `int` thuần của Python.

In [13]:
def size_shape(players: pd.DataFrame) -> List[int]:
    return list(players.shape)

out = size_shape(players)
print(out, type(out), [type(v).__name__ for v in out])

[5, 5] <class 'list'> ['int', 'int']


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def getDataframeSize(players: pd.DataFrame) -> List[int]:
    return list(players.shape)

print(getDataframeSize(players))

[5, 5]


**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

---

### Cách 4 — Đọc từng phần tử của `shape`, hoặc `index.size` / `columns.size`

**Giải thích.** `players.shape[0]` là số hàng, `players.shape[1]` là số cột. Tương đương: `players.index.size` và `players.columns.size`. Cách viết này rõ ràng "đâu là hàng, đâu là cột".

In [15]:
def size_indexed(players: pd.DataFrame) -> List[int]:
    rows, cols = players.shape            # giải nén tuple
    return [rows, cols]

def size_axes_size(players: pd.DataFrame) -> List[int]:
    return [players.index.size, players.columns.size]

print(size_indexed(players), size_axes_size(players))

[5, 5] [5, 5]


In [16]:
# Code nộp LeetCode cho Cách 4
import pandas as pd

def getDataframeSize(players: pd.DataFrame) -> List[int]:
    rows, cols = players.shape
    return [rows, cols]

print(getDataframeSize(players))

[5, 5]


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

---

### Cách 5 — `axes` hoặc `to_numpy().shape`

**Giải thích.**

- `[len(ax) for ax in players.axes]` lấy độ dài hai trục `[index, columns]`.
- `players.to_numpy().shape` đổi bảng sang mảng NumPy rồi lấy `shape`. Cách này **tạo bản sao dữ liệu** nên tốn `O(m·c)`: **không nên** dùng, chỉ để thấy "một cách khác" cũng cho cùng kết quả.
- Dùng `size`: `rows = size // cols` (chỉ khi `cols > 0`).

In [17]:
def size_axes(players: pd.DataFrame) -> List[int]:
    return [len(ax) for ax in players.axes]

def size_numpy(players: pd.DataFrame) -> List[int]:
    return list(players.to_numpy().shape)

def size_by_size(players: pd.DataFrame) -> List[int]:
    cols = len(players.columns)
    return [players.size // cols if cols else 0, cols]

print(size_axes(players), size_numpy(players), size_by_size(players))

[5, 5] [5, 5] [5, 5]


**Độ phức tạp Cách 5:** `axes` là `O(1)`; `to_numpy().shape` là `O(m·c)` Time và Space; `size // cols` là `O(1)`. Cách `size // cols` hỏng khi `cols = 0` (chia cho 0) nên ta chặn riêng.

## 8. TRACE từng bước

Phần này in **`shape` qua từng bước biến đổi** của DataFrame. Mỗi phép biến đổi làm hình dạng thay đổi theo quy luật dễ đoán.

In [18]:
def trace_shapes(players):
    steps = [
        ("bảng gốc", players),
        ("head(2): giữ 2 hàng đầu", players.head(2)),
        ("chọn 2 cột name, age", players[["name", "age"]]),
        ("bỏ cột age (drop)", players.drop(columns="age")),
        ("lọc age >= 30", players[players["age"] >= 30]),
        ("chuyển vị .T (đổi hàng <-> cột)", players.T),
    ]
    for label, df in steps:
        print(f"--- {label}: shape = {df.shape} -> {list(df.shape)}")
        print(df.to_string()[:300])
        print()

trace_shapes(players)

--- bảng gốc: shape = (5, 5) -> [5, 5]
   player_id      name  age    position               team
0        846     Mason   21     Forward        Real Madrid
1        749     Riley   30      Winger          Barcelona
2        155       Bob   28     Striker  Manchester United
3        426  Isabella   32  Goalkeeper          Liverpool
4    

--- head(2): giữ 2 hàng đầu: shape = (2, 5) -> [2, 5]
   player_id   name  age position         team
0        846  Mason   21  Forward  Real Madrid
1        749  Riley   30   Winger    Barcelona

--- chọn 2 cột name, age: shape = (5, 2) -> [5, 2]
       name  age
0     Mason   21
1     Riley   30
2       Bob   28
3  Isabella   32
4      Zane   36

--- bỏ cột age (drop): shape = (5, 4) -> [5, 4]
   player_id      name    position               team
0        846     Mason     Forward        Real Madrid
1        749     Riley      Winger          Barcelona
2        155       Bob     Striker  Manchester United
3        426  Isabella  Goalkeeper          L

In [19]:
def trace_counting(players):
    rows = 0
    print("Đếm hàng:")
    for idx, row in enumerate(players.itertuples(index=False), start=1):
        rows += 1
        print(f"  hàng {idx}: {row.name}  -> rows = {rows}")
    cols = 0
    print("Đếm cột:")
    for name in players.columns:
        cols += 1
        print(f"  cột {cols}: {name}  -> cols = {cols}")
    print("kết quả:", [rows, cols])
    return [rows, cols]

trace_counting(players)

Đếm hàng:
  hàng 1: Mason  -> rows = 1
  hàng 2: Riley  -> rows = 2
  hàng 3: Bob  -> rows = 3
  hàng 4: Isabella  -> rows = 4
  hàng 5: Zane  -> rows = 5
Đếm cột:
  cột 1: player_id  -> cols = 1
  cột 2: name  -> cols = 2
  cột 3: age  -> cols = 3
  cột 4: position  -> cols = 4
  cột 5: team  -> cols = 5
kết quả: [5, 5]


[5, 5]

Quy luật rút ra từ bảng trace: `head(2)` giữ nguyên số cột; chọn / bỏ cột giữ nguyên số hàng; lọc theo điều kiện giữ nguyên số cột; **chuyển vị đổi chỗ** hàng và cột (`(5, 5)` vẫn vuông nên không thấy, hãy thử `players.head(2).T.shape = (5, 2)`).

In [20]:
print(players.head(2).shape, "->", players.head(2).T.shape)    # (2, 5) -> (5, 2)

(2, 5) -> (5, 2)


## 9. Code nộp LeetCode

In [21]:
import pandas as pd
from typing import List

def getDataframeSize(players: pd.DataFrame) -> List[int]:
    return list(players.shape)

print(getDataframeSize(players))

[5, 5]


## 10. Vì sao thuật toán đúng

- **Định nghĩa:** theo tài liệu pandas, `df.shape` là tuple biểu diễn **số hàng** (độ dài `index`) và **số cột** (độ dài `columns`) theo đúng thứ tự `(hàng, cột)`.
- **Không đếm sót:** `shape` lấy thẳng từ cấu trúc nội bộ, không phụ thuộc vào việc ô có `NaN` hay không (khác với `count()`).
- **Index không phải cột:** nhãn hàng (`index`) không được tính vào số cột; tiêu đề cột không được tính vào số hàng — đúng với đề.
- **Kiểu trả về:** `list(...)` biến tuple thành list gồm hai số nguyên Python, khớp `List[int]`.
- **Biên:** bảng không có hàng cho `[0, c]`; bảng không có cột cho `[m, 0]`; bảng rỗng `[0, 0]` — đều đúng theo `shape`.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` là số cột.

| Cách | Time | Space |
|---|---|---|
| 1. Đếm thủ công | `O(m + c)` (kèm tạo namedtuple mỗi hàng) | `O(1)` |
| 2. `len` hai lần | `O(1)` | `O(1)` |
| 3. `list(df.shape)` | `O(1)` | `O(1)` |
| 4. Giải nén `shape` / `index.size` | `O(1)` | `O(1)` |
| 5a. `axes` | `O(1)` | `O(1)` |
| 5b. `to_numpy().shape` | `O(m·c)` | `O(m·c)` |

Kích thước bảng là **siêu dữ liệu đã lưu sẵn**, đọc ra là hằng số. Cách 1 và 5b chậm hơn nhiều khi bảng lớn.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [22]:
impls = {
    "counting": size_by_counting,
    "len": size_len,
    "shape": size_shape,
    "indexed": size_indexed,
    "index_size": size_axes_size,
    "axes": size_axes,
    "numpy": size_numpy,
    "submit": getDataframeSize,
}

def make(r, c, nan=False):
    arr = np.arange(r * c, dtype="float64").reshape(r, c)
    if nan and arr.size:
        arr[0, 0] = np.nan
    return pd.DataFrame(arr, columns=[f"c{j}" for j in range(c)])

cases = [
    ("ví dụ của đề", players, [5, 5]),
    ("3 hàng, 2 cột", pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]}), [3, 2]),
    ("một hàng một cột", make(1, 1), [1, 1]),
    ("một cột nhiều hàng", make(10, 1), [10, 1]),
    ("một hàng nhiều cột", make(1, 10), [1, 10]),
    ("có NaN", make(4, 3, nan=True), [4, 3]),
    ("không hàng, có cột", pd.DataFrame(columns=["a", "b", "c"]), [0, 3]),
]
for name, df, exp in cases:
    for key, fn in impls.items():
        got = fn(df)
        assert got == exp, (name, key, got, exp)
        assert isinstance(got, list) and all(type(v) is int for v in got), (name, key, got)
    print("PASS", name, "->", exp)

PASS ví dụ của đề -> [5, 5]
PASS 3 hàng, 2 cột -> [3, 2]
PASS một hàng một cột -> [1, 1]
PASS một cột nhiều hàng -> [10, 1]
PASS một hàng nhiều cột -> [1, 10]
PASS có NaN -> [4, 3]
PASS không hàng, có cột -> [0, 3]


Chú ý: ca "không cột" (`pd.DataFrame()`) cho `[0, 0]`; riêng `size_by_size` đã chặn chia cho 0.

In [23]:
assert getDataframeSize(pd.DataFrame()) == [0, 0]
assert size_by_size(pd.DataFrame()) == [0, 0]
assert size_by_size(players) == [5, 5]
print("PASS bảng rỗng hoàn toàn")

PASS bảng rỗng hoàn toàn


### 12.2. Phản ví dụ cho các lỗi hay gặp

Dùng bảng **không vuông** để lộ lỗi đảo hàng ↔ cột, lỗi `size`, lỗi `count()`.

In [24]:
t = make(7, 3, nan=True)                      # 7 hàng, 3 cột, một ô NaN
assert getDataframeSize(t) == [7, 3]
assert t.size == 21                           # size = 7 * 3, KHÔNG phải số hàng
assert t.count().tolist() == [6, 7, 7]        # count() bỏ qua NaN
assert tuple(getDataframeSize(t)) == t.shape  # list vs tuple: so sánh sau khi đổi kiểu
assert getDataframeSize(t) != t.shape         # list != tuple
assert getDataframeSize(t) != [3, 7]          # đảo hàng/cột là sai
print("PASS phản ví dụ")

PASS phản ví dụ


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh DataFrame ngẫu nhiên với `r ∈ [0, 60]` hàng, `c ∈ [1, 12]` cột (có thể kèm `NaN`), so mọi cách với kích thước đã biết.

In [25]:
import random
random.seed(2878)
np.random.seed(2878)
for t in range(300):
    r = random.randint(0, 60)
    c = random.randint(1, 12)
    arr = np.random.rand(r, c)
    if r and random.random() < 0.5:
        arr[np.random.randint(r), np.random.randint(c)] = np.nan
    df = pd.DataFrame(arr, columns=[f"c{j}" for j in range(c)])
    for key, fn in impls.items():
        assert fn(df) == [r, c], (t, key)
print("300 DataFrame ngẫu nhiên: mọi cách đều cho [hàng, cột] đúng")

300 DataFrame ngẫu nhiên: mọi cách đều cho [hàng, cột] đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về tuple `shape` thay vì list

In [26]:
def sai_tuple(players):
    return players.shape                       # sai: tuple (5, 5)

print("sai :", sai_tuple(players), type(sai_tuple(players)).__name__)
print("sai  == [5, 5] ?", sai_tuple(players) == [5, 5])        # False
print("đúng:", getDataframeSize(players), getDataframeSize(players) == [5, 5])

sai : (5, 5) tuple
sai  == [5, 5] ? False
đúng: [5, 5] True


**Vì sao sai:** đề yêu cầu **list**. Trong Python `(5, 5) == [5, 5]` là `False`, nên bộ chấm có thể không nhận. Dùng `list(players.shape)`.

### Lỗi 2: Đảo thứ tự cột và hàng

In [27]:
def sai_dao(players):
    return [len(players.columns), len(players)]        # sai: [cột, hàng]

t = make(7, 3)
print("sai :", sai_dao(t))                # [3, 7]
print("đúng:", getDataframeSize(t))       # [7, 3]

sai : [3, 7]
đúng: [7, 3]


**Vì sao sai:** thứ tự đề là `[số hàng, số cột]`. Bảng vuông (5×5) **che** lỗi này; luôn thử thêm bảng không vuông.

### Lỗi 3: Dùng `df.size`

In [28]:
def sai_size(players):
    return [players.size]                    # sai: size = số ô

print("sai :", sai_size(players))           # [25]
print("đúng:", getDataframeSize(players))   # [5, 5]

sai : [25]
đúng: [5, 5]


**Vì sao sai:** `size` là **tổng số ô** (`hàng × cột`) — một số duy nhất, không phải hai số.

### Lỗi 4: Dùng `count()` để đếm hàng

In [29]:
def sai_count(players):
    return [players.count().iloc[0], len(players.columns)]   # sai khi cột đầu có NaN

t = make(5, 3, nan=True)                    # ô [0, 0] là NaN
print("sai :", sai_count(t))                # [4, 3]
print("đúng:", getDataframeSize(t))         # [5, 3]

sai : [np.int64(4), 3]
đúng: [5, 3]


**Vì sao sai:** `count()` **bỏ qua `NaN`**, nên cho số hàng nhỏ hơn thực tế. Muốn số hàng dùng `len(df)` hoặc `df.shape[0]`.

### Lỗi 5: Tưởng cột index / dòng tiêu đề là dữ liệu

In [30]:
print(players.shape[1], "cột dữ liệu (không tính index)")
print(len(players.reset_index().columns), "cột nếu reset_index() thêm cột 'index'")

5 cột dữ liệu (không tính index)
6 cột nếu reset_index() thêm cột 'index'


**Vì sao sai:** index là nhãn hàng, không phải một cột; `players.shape` đã không tính nó. Nếu bạn gọi `reset_index()` thì index biến thành cột thật và số cột tăng 1 — đừng làm như vậy trước khi lấy kích thước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Kích thước / số hàng / số cột" → `df.shape` (hoặc `len(df)`).
- Nhớ câu: "**shape = (hàng, cột)**" và "**size = hàng × cột**".
- Đề đòi `List[int]` thì đổi tuple: `list(df.shape)`.
- Số hàng thật: `len(df)`; không dùng `count()` khi có thể có `NaN`.
- Khi nghi ngờ hình dạng sau biến đổi: in `df.shape` **sau từng bước**, như bảng trace ở mục 8.
- Kiểm tra bằng bảng **không vuông** để phát hiện lỗi đảo trục.

## 15. Biến thể của bài

1. **Chỉ lấy số hàng:** `len(df)` hoặc `df.shape[0]`.
2. **Số ô không thiếu:** `df.count().sum()`; **số ô thiếu:** `df.isna().sum().sum()`.
3. **Kích thước theo nhóm:** `df.groupby("team").size()`.
4. **Kích thước bộ nhớ:** `df.memory_usage(deep=True).sum()`.
5. **Tên và kiểu cột:** `df.dtypes`; liên quan LeetCode 2879 *Display the First Three Rows* và 2880 *Select Data*.
6. **Hình dạng sau phép lọc:** đếm số hàng thỏa điều kiện `len(df[df["age"] > 30])`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm thủ công | vòng lặp đếm hàng, cột | `O(m + c)` | `O(1)` | chỉ để hiểu khái niệm |
| 2. `len` hai lần | `[len(df), len(df.columns)]` | `O(1)` | `O(1)` | rất rõ ràng |
| 3. `list(df.shape)` | đổi tuple thành list | `O(1)` | `O(1)` | **cách nộp bài** |
| 4. Giải nén `shape` | `rows, cols = df.shape` | `O(1)` | `O(1)` | khi cần dùng riêng từng số |
| 5. `axes` / `to_numpy` | độ dài hai trục / mảng NumPy | `O(1)` / `O(m·c)` | `O(1)` / `O(m·c)` | `axes` ổn; `to_numpy` nên tránh |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]}).shape`, `pd.DataFrame({"a": [1, 2, 3]}).T.shape`, `pd.DataFrame(np.zeros((4, 6))).size`, `(2, 3) == [2, 3]`.

**Bài 2 (tính tay).** Một bảng có các cột `id`, `name`, `score` và 4 hàng dữ liệu. Sau lần lượt: `df.drop(columns="score")`, rồi `.head(3)`, rồi `.T`. Viết `shape` sau mỗi bước.

**Bài 3 (điền TODO).** Hoàn thành `get_size_todo(df)` bằng giải nén `shape`.

In [31]:
def get_size_todo(df):
    # TODO 1: rows, cols = df.shape
    # TODO 2: return [rows, cols]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `get_size_len(df)` chỉ bằng `len` mà không nhìn code mẫu. Kiểm tra với ví dụ của đề và với bảng `7 × 3`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_missing_cells(df)` trả về **số ô `NaN`** của bảng. (b) Viết `size_after_dropna(df)`: kích thước `[hàng, cột]` sau khi **bỏ mọi hàng có `NaN`** (`dropna()`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(3, 2)`; `(1, 3)`; `24`; `False` (tuple khác list). Bài 2: gốc `(4, 3)`.
- **Mức 2 (vừa):** Bài 2: `drop` bớt một cột thành `(4, 2)`; `head(3)` thành `(3, 2)`; `.T` thành `(2, 3)`. Bài 3: hai dòng đã ghi trong comment. Bài 4: `[len(df), len(df.columns)]`.
- **Mức 3 (sâu):** Bài 5(a): `df.isna().sum().sum()`: `isna()` cho bảng True/False cùng hình dạng, `sum()` cộng từng cột, `sum()` lần hai cộng các cột. Bài 5(b): `list(df.dropna().shape)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]}).shape == (3, 2)
assert pd.DataFrame({"a": [1, 2, 3]}).T.shape == (1, 3)
assert pd.DataFrame(np.zeros((4, 6))).size == 24
assert ((2, 3) == [2, 3]) is False

# Bài 2
b2 = pd.DataFrame({"id": [1, 2, 3, 4], "name": list("abcd"), "score": [5, 6, 7, 8]})
assert b2.shape == (4, 3)
s1 = b2.drop(columns="score"); assert s1.shape == (4, 2)
s2 = s1.head(3);               assert s2.shape == (3, 2)
s3 = s2.T;                     assert s3.shape == (2, 3)

# Bài 3
def get_size_todo_ans(df):
    rows, cols = df.shape
    return [rows, cols]

assert get_size_todo_ans(players) == [5, 5]
assert get_size_todo_ans(make(7, 3)) == [7, 3]

# Bài 4
def get_size_len(df):
    return [len(df), len(df.columns)]

assert get_size_len(players) == [5, 5]
assert get_size_len(make(7, 3)) == [7, 3]

# Bài 5(a)
def count_missing_cells(df):
    return int(df.isna().sum().sum())

assert count_missing_cells(make(4, 3, nan=True)) == 1
assert count_missing_cells(players) == 0

# Bài 5(b)
def size_after_dropna(df):
    return list(df.dropna().shape)

assert size_after_dropna(make(4, 3, nan=True)) == [3, 3]
assert size_after_dropna(players) == [5, 5]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lấy kích thước DataFrame)

```python
import pandas as pd

def getDataframeSize(players: pd.DataFrame) -> List[int]:
    return list(players.shape)      # [số hàng, số cột]
```

**Checklist**

- [ ] Nhớ: `shape = (hàng, cột)`; hàng trước, cột sau?
- [ ] Trả về **list** (`list(df.shape)`), không phải tuple?
- [ ] Không nhầm với `df.size` (tổng số ô) hay `count()` (bỏ qua `NaN`)?
- [ ] Đã thử bảng không vuông, bảng có `NaN`, bảng không hàng?
- [ ] Không gọi `reset_index()` hay thay đổi bảng trước khi đo kích thước?
- [ ] Skeleton đúng `def getDataframeSize(players: pd.DataFrame) -> List[int]` (đối chiếu lại với đề gốc nếu lệch)?